In [1]:
import numpy as np

In [3]:
def load_data(path, isImage):

    with open(path, 'rb') as f:
        data = f.read()
        
    magic = int.from_bytes(data[0:4], 'big')
    num_images = int.from_bytes(data[4:8], 'big')
    rows = int.from_bytes(data[8:12], 'big')
    cols = int.from_bytes(data[12:16], 'big')
    
    if isImage:
        offset = 16  # images have 16-byte header
        arr = np.frombuffer(data, dtype=np.uint8, offset=offset)
        arr = arr.reshape(num_images, rows, cols)
    else:
        offset = 8  # labels have 8-byte header
        arr = np.frombuffer(data, dtype=np.uint8, offset=offset)
        
    return arr


def load_mnist():
    x_train = load_data("../../data/mnist_dataset/train-images-idx3-ubyte", True)
    y_train = load_data("../../data/mnist_dataset/train-labels-idx1-ubyte", False)
    x_test  = load_data("../../data/mnist_dataset/t10k-images-idx3-ubyte", True)
    y_test  = load_data("../../data/mnist_dataset/t10k-labels-idx1-ubyte", False)
    return x_train, y_train, x_test, y_test

In [5]:
x_train, y_train, x_test, y_test = load_mnist()

In [6]:
def logistic_regression_k_folds(x_train, y_train, model_digit_c1, model_digit_c2,
                             k=5, batch_size=32, learning_rate=0.01,
                             validation_patience=5, epsilon=1e-4):

    # --- 1. Normalize and filter to the two classes ONCE up front ---
    x = x_train / 255.0

    mask = (y_train == model_digit_c1) | (y_train == model_digit_c2)
    x = x[mask]
    y = (y_train[mask] == model_digit_c1).astype(float).reshape(-1, 1)

    N = x.shape[0]

    # --- 2. Shuffle once before folding ---
    indices = np.random.permutation(N)
    x = x[indices]
    y = y[indices]

    # --- 3. Split into k roughly-equal folds ---
    fold_indices = np.array_split(np.arange(N), k)

    fold_weights = []
    fold_val_losses = []

    for fold_idx in range(k):
        print(f"\n{'='*40}")
        print(f"Fold {fold_idx + 1} / {k}")
        print(f"{'='*40}")

        # --- 4. Build train / val splits for this fold ---
        val_idx   = fold_indices[fold_idx]
        train_idx = np.concatenate([fold_indices[i] for i in range(k) if i != fold_idx])

        x_val_fold   = x[val_idx]
        y_val_fold   = y[val_idx]
        x_train_fold = x[train_idx]
        y_train_fold = y[train_idx]

        # --- 5. Flatten + add bias column ---
        x_train_flat = x_train_fold.reshape(x_train_fold.shape[0], -1)
        x_train_bias = np.hstack([x_train_flat, np.ones((x_train_flat.shape[0], 1))])

        x_val_flat   = x_val_fold.reshape(x_val_fold.shape[0], -1)
        x_val_bias   = np.hstack([x_val_flat, np.ones((x_val_flat.shape[0], 1))])

        # --- 6. Class weights (same logic as original) ---
        n_fold   = x_train_bias.shape[0]
        n_pos    = np.sum(y_train_fold == 1)
        n_neg    = n_fold - n_pos
        w_pos    = n_fold / (2 * n_pos)
        w_neg    = n_fold / (2 * n_neg)

        # --- 7. Train using the same loop as your original function ---
        w = np.random.uniform(low=-0.01, high=0.01, size=(785, 1))
        best_val_loss = float('inf')
        best_w  = w.copy()
        counter = 0
        epoch   = 0
        max_epochs = 100

        while epoch < max_epochs:
            perm = np.random.permutation(n_fold)
            x_shuf = x_train_bias[perm]
            y_shuf = y_train_fold[perm]

            for start in range(0, n_fold, batch_size):
                end      = min(start + batch_size, n_fold)
                xi_batch = x_shuf[start:end]
                yi_batch = y_shuf[start:end]

                z        = np.dot(xi_batch, w)
                y_hat    = 1 / (1 + np.exp(-z))

                sample_weights = yi_batch * w_pos + (1 - yi_batch) * w_neg
                error    = (y_hat - yi_batch) * sample_weights
                de_dw    = np.dot(xi_batch.T, error) / xi_batch.shape[0]
                w       -= learning_rate * de_dw

            # Validation loss for early stopping
            z_v      = np.dot(x_val_bias, w)
            y_hat_v  = np.clip(1 / (1 + np.exp(-z_v)), 1e-15, 1 - 1e-15)
            v_loss   = -np.mean(y_val_fold * np.log(y_hat_v) +
                                (1 - y_val_fold) * np.log(1 - y_hat_v))

            if v_loss < best_val_loss - epsilon:
                best_val_loss = v_loss
                best_w  = w.copy()
                counter = 0
            else:
                counter += 1

            print(f"Epoch {epoch:3d} | val_loss: {v_loss:.5f}")

            if counter == validation_patience:
                print(f"Early stopping at epoch {epoch}")
                break

            epoch += 1
            if epoch == max_epochs:
                print("Reached max epochs.")

In [24]:
from itertools import combinations

weights_dict = {}

digits = list(range(10))  # 0 → 9

for c1, c2 in combinations(digits, 2):
    print(f"Training model for {c1} vs {c2}")
    
    w = logistic_regression_k_folds(
        x_train, y_train,
        c1, c2,
        k=5,
        batch_size=1,
        learning_rate=0.01,
        validation_patience=5,
        epsilon=1e-4
    )
    
    weights_dict[(c1, c2)] = w

Training model for 0 vs 1

Fold 1 / 5
Epoch   0 | val_loss: 0.00813
Epoch   1 | val_loss: 0.00708
Epoch   2 | val_loss: 0.00709
Epoch   3 | val_loss: 0.00691
Epoch   4 | val_loss: 0.00677
Epoch   5 | val_loss: 0.00682
Epoch   6 | val_loss: 0.00682
Epoch   7 | val_loss: 0.00684
Epoch   8 | val_loss: 0.00707
Epoch   9 | val_loss: 0.00690
Early stopping at epoch 9

Fold 2 / 5
Epoch   0 | val_loss: 0.00655
Epoch   1 | val_loss: 0.00429
Epoch   2 | val_loss: 0.00360
Epoch   3 | val_loss: 0.00310
Epoch   4 | val_loss: 0.00288
Epoch   5 | val_loss: 0.00267
Epoch   6 | val_loss: 0.00271
Epoch   7 | val_loss: 0.00251
Epoch   8 | val_loss: 0.00239
Epoch   9 | val_loss: 0.00241
Epoch  10 | val_loss: 0.00252
Epoch  11 | val_loss: 0.00234
Epoch  12 | val_loss: 0.00235
Epoch  13 | val_loss: 0.00227
Epoch  14 | val_loss: 0.00212
Epoch  15 | val_loss: 0.00216
Epoch  16 | val_loss: 0.00221
Epoch  17 | val_loss: 0.00205
Epoch  18 | val_loss: 0.00209
Epoch  19 | val_loss: 0.00209
Early stopping at epoch 

In [7]:
def evaluate_digit_ovo(weights_dict, x_test, y_test, target_digit):
    
    # ---- Step 1: Predict all ----
    def predict_one(x):
        votes = {i: 0 for i in range(10)}

        x_flat = x.reshape(1, -1) / 255.0
        x_bias = np.hstack([x_flat, np.ones((1, 1))])

        for (c1, c2), w in weights_dict.items():
            z = np.dot(x_bias, w)
            y_hat = 1 / (1 + np.exp(-z))

            if y_hat >= 0.5:
                votes[c1] += 1
            else:
                votes[c2] += 1

        return max(votes, key=votes.get)

    y_pred = np.array([predict_one(x) for x in x_test])

    # ---- Step 2: Convert to binary (target vs all) ----
    y_true_bin = (y_test == target_digit).astype(int)
    y_pred_bin = (y_pred == target_digit).astype(int)

    # ---- Step 3: Compute metrics ----
    tp = np.sum((y_pred_bin == 1) & (y_true_bin == 1))
    tn = np.sum((y_pred_bin == 0) & (y_true_bin == 0))
    fp = np.sum((y_pred_bin == 1) & (y_true_bin == 0))
    fn = np.sum((y_pred_bin == 0) & (y_true_bin == 1))

    accuracy  = (tp + tn) / len(y_test)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # ---- Step 4: Print ----
    print(f"Results for digit {target_digit} vs All")
    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

def logistic_regression_k_folds(x_train, y_train, model_digit_c1, model_digit_c2,
                             k=5, batch_size=32, learning_rate=0.01,
                             validation_patience=5, epsilon=1e-4):

    # --- 1. Normalize and filter to the two classes ONCE up front ---
    x = x_train / 255.0

    mask = (y_train == model_digit_c1) | (y_train == model_digit_c2)
    x = x[mask]
    y = (y_train[mask] == model_digit_c1).astype(float).reshape(-1, 1)

    N = x.shape[0]

    # --- 2. Shuffle once before folding ---
    indices = np.random.permutation(N)
    x = x[indices]
    y = y[indices]

    # --- 3. Split into k roughly-equal folds ---
    fold_indices = np.array_split(np.arange(N), k)

    fold_weights = []
    fold_val_losses = []

    for fold_idx in range(k):
        print(f"\n{'='*40}")
        print(f"Fold {fold_idx + 1} / {k}")
        print(f"{'='*40}")

        # --- 4. Build train / val splits for this fold ---
        val_idx   = fold_indices[fold_idx]
        train_idx = np.concatenate([fold_indices[i] for i in range(k) if i != fold_idx])

        x_val_fold   = x[val_idx]
        y_val_fold   = y[val_idx]
        x_train_fold = x[train_idx]
        y_train_fold = y[train_idx]

        # --- 5. Flatten + add bias column ---
        x_train_flat = x_train_fold.reshape(x_train_fold.shape[0], -1)
        x_train_bias = np.hstack([x_train_flat, np.ones((x_train_flat.shape[0], 1))])

        x_val_flat   = x_val_fold.reshape(x_val_fold.shape[0], -1)
        x_val_bias   = np.hstack([x_val_flat, np.ones((x_val_flat.shape[0], 1))])

        # --- 6. Class weights (same logic as original) ---
        n_fold   = x_train_bias.shape[0]
        n_pos    = np.sum(y_train_fold == 1)
        n_neg    = n_fold - n_pos
        w_pos    = n_fold / (2 * n_pos)
        w_neg    = n_fold / (2 * n_neg)

        # --- 7. Train using the same loop as your original function ---
        w = np.random.uniform(low=-0.01, high=0.01, size=(785, 1))
        best_val_loss = float('inf')
        best_w  = w.copy()
        counter = 0
        epoch   = 0
        max_epochs = 100

        while epoch < max_epochs:
            perm = np.random.permutation(n_fold)
            x_shuf = x_train_bias[perm]
            y_shuf = y_train_fold[perm]

            for start in range(0, n_fold, batch_size):
                end      = min(start + batch_size, n_fold)
                xi_batch = x_shuf[start:end]
                yi_batch = y_shuf[start:end]

                z        = np.dot(xi_batch, w)
                y_hat    = 1 / (1 + np.exp(-z))

                sample_weights = yi_batch * w_pos + (1 - yi_batch) * w_neg
                error    = (y_hat - yi_batch) * sample_weights
                de_dw    = np.dot(xi_batch.T, error) / xi_batch.shape[0]
                w       -= learning_rate * de_dw

            # Validation loss for early stopping
            z_v      = np.dot(x_val_bias, w)
            y_hat_v  = np.clip(1 / (1 + np.exp(-z_v)), 1e-15, 1 - 1e-15)
            v_loss   = -np.mean(y_val_fold * np.log(y_hat_v) +
                                (1 - y_val_fold) * np.log(1 - y_hat_v))

            if v_loss < best_val_loss - epsilon:
                best_val_loss = v_loss
                best_w  = w.copy()
                counter = 0
            else:
                counter += 1

            print(f"Epoch {epoch:3d} | val_loss: {v_loss:.5f}")

            if counter == validation_patience:
                print(f"Early stopping at epoch {epoch}")
                break

            epoch += 1
            if epoch == max_epochs:
                print("Reached max epochs.")
        
        fold_weights.append(best_w)
        fold_val_losses.append(best_val_loss)
    
    # Average weights across all folds
    avg_w = np.mean(fold_weights, axis=0)
    return avg_w